# Dark Web Credential Engine — Exploration Notebook

Use this notebook to explore data, tune features, and visualize model results.


In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.ingestion.data_generator import generate_breach_dataset, generate_internal_employee_dataset
from src.preprocessing.cleaner import BreachDataCleaner
from src.nlp.entity_extractor import EntityExtractor
from src.correlation.correlator import CredentialCorrelator
from src.ml.risk_scorer import RiskScorer, engineer_features, FEATURE_COLUMNS

print('All imports OK')

## 1. Generate and Explore Raw Breach Data

In [ ]:
breach_raw = generate_breach_dataset(500)
print(f'Shape: {breach_raw.shape}')
breach_raw.head()

In [ ]:
# Distribution of breach sources
breach_raw['breach_source'].value_counts().plot(kind='bar', figsize=(10,4), title='Breach Source Distribution')
plt.tight_layout(); plt.show()

## 2. Preprocessing Stats

In [ ]:
cleaner = BreachDataCleaner(config_path='../configs/config.yaml')
breach_clean = cleaner.run(breach_raw)
print('Preprocessing stats:', cleaner.get_stats())
breach_clean.head()

## 3. NLP Entity Extraction

In [ ]:
extractor = EntityExtractor(config_path='../configs/config.yaml')

# Test on a single string
sample = 'admin@acme-corp.com:password123'
print('Single extraction:', extractor.extract_from_text(sample))

# Run on full dataset
breach_nlp = extractor.extract_from_dataframe(breach_clean)
print('\nKeyword stats:', extractor.get_keyword_stats(breach_nlp))

## 4. Credential Correlation

In [ ]:
employee_df = generate_internal_employee_dataset(100)
correlator  = CredentialCorrelator(config_path='../configs/config.yaml')
correlated  = correlator.correlate(breach_nlp, employee_df)

print(f"Compromised: {correlated['is_compromised'].sum()} / {len(correlated)}")
correlated[correlated['is_compromised']][['full_name','email','department','breach_count','match_types']].head(10)

## 5. Feature Engineering & ML

In [ ]:
# Engineer features
feat_df = engineer_features(correlated)
feat_df[FEATURE_COLUMNS].describe()

In [ ]:
# Feature correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(feat_df[FEATURE_COLUMNS].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature Correlation Matrix')
plt.tight_layout(); plt.show()

In [ ]:
# Train and evaluate model
scorer  = RiskScorer(config_path='../configs/config.yaml')
metrics = scorer.train(correlated)
print('Accuracy:', metrics['accuracy'])
print('ROC AUC: ', metrics['roc_auc'])
print('\nClassification Report:')
print(metrics['classification_report'])

In [ ]:
# Score all employees
risk_df   = scorer.score(correlated)
risk_df   = scorer.explain(risk_df)

# Risk score distribution
risk_df['risk_score'].hist(bins=20, figsize=(10,4), color='steelblue', edgecolor='white')
plt.axvline(60, color='red', linestyle='--', label='Threshold (60)')
plt.title('Risk Score Distribution'); plt.xlabel('Risk Score'); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Feature importance
if scorer.feature_importance:
    imp = pd.Series(scorer.feature_importance)
    imp.sort_values().plot(kind='barh', figsize=(8,5), title='Feature Importance')
    plt.tight_layout(); plt.show()

In [ ]:
# SHAP explanation (if shap installed)
try:
    import shap
    shap_vals = scorer.get_shap_values(correlated)
    if shap_vals is not None:
        from src.ml.risk_scorer import engineer_features, FEATURE_COLUMNS
        X = engineer_features(correlated)[FEATURE_COLUMNS].fillna(0)
        X_scaled = scorer.scaler.transform(X)
        shap.summary_plot(shap_vals[1], X_scaled, feature_names=FEATURE_COLUMNS)
except ImportError:
    print('Install shap: pip install shap')

## 6. Top High-Risk Employees

In [ ]:
top_risk = risk_df.nlargest(10, 'risk_score')[[
    'full_name','email','department','role','risk_score','risk_level',
    'breach_count','breach_sources','risk_explanation'
]]
top_risk